<small>CONFIDENTIAL PROPERTY OF MOUNTAIN TOP BRAND LLC</small><br>
<small>Author: Jeremy Barton</small><br>
<small>Created: 09-14-2025</small><br>
<small>Updated: 10-05-2025</small>


# Data Colletion Techniques

### Appendix

1. [Product Page Scrubbing](#product-page-scrubbing)
2. [Article Keyword Detection](#article-keyword-detection) 



### Introduction

This notebook will cover three methods Mountain Top Brand will use to collect data on the open web in order to assist with category selection, trend tracking, and building a database of detailed marketplace product and pricing information. 

These three methods are essentially algorithms which together form a foundation for AI programs to work from. 

<hr>

## Product Page Scrubbing

Business Requirements

What if you had somebody working for the Company whose job it was to scroll around product pages of Amazon and record the prices of each competing item in your selected category? That is what this method does, but with as many employees as the target servers won't detect as spam.

Technical Requirements

The below code reads in data collected from running file in `amzn-scraper/timing.py`. It scans for ASIN codes which specify particular product pages. ASIN codes link to specific product pages for an item on Amazon. Through the product page, details can be scrapted and harvested for analysis.

In [1]:
import json
import pandas as pd
from pandas import json_normalize

# Read the JSON file first
with open('/home/jeremy/Documents/GitHub/mtb-data-research/price_data/all_prices.json', 'r') as f:
    data = json.load(f)

#### Randomization Strategy

To circumvent Amazon's bot detection a randomization strategy is deployed to sucessfully extract product detail and price data.

There are 4 time windows.

<ol><li>Morning: 6 AM - 9 AM (3-hour window)
    <li>Noon: 11 AM - 1 PM (2-hour window)
    <li>Evening: 5 PM - 8 PM (3-hour window)
    <li>Overnight: 12 AM - 3 AM (3-hour window)
</ol>

Each ASIN code is schedulef=d to be pinged once in each time window. If 5 ASINs are pinged, there will be 20 total runs per day. (5 ASINS x4 windows). If the randomly selected time has already passed today, it automatically schedules for the same time tomorrow.

#### Data Example

In [2]:
# Normalize the 'data' array and add the top-level timestamp
listing_data = json_normalize(data['data'])
listing_data['main_timestamp'] = data['timestamp']

# Convert timestamps to datetime
listing_data['timestamp'] = pd.to_datetime(listing_data['timestamp'])
listing_data['main_timestamp'] = pd.to_datetime(listing_data['main_timestamp'])

listing_data

,asin,timestamp,price,raw_price,main_timestamp
0,B08573DQ39,2025-05-09 19:38:33.414013,21.99,$21.99,2025-05-09 19:38:40.180263
1,B0BY897GH4,2025-05-09 19:38:36.922746,23.99,$23.99,2025-05-09 19:38:40.180263
2,B07HGCJB1F,2025-05-09 19:38:40.180207,21.99,$21.99,2025-05-09 19:38:40.180263


<hr>

## Article Keyword Detection

Business Requirements

Whether we like it or not, the news influences public sentiment. This can expose the direction consumers mindset may start to shift, which can impact buying decicions. News sources offer RSS feeds that programmically generate links to article content. This can be scrubbed to MTB servers to extract keywords. There is a list of hundreds of RSS feeds that the Company can use to collection this data. It is located <a src="/home/jeremy/Documents/GitHub/mtb-data-research/rss-urls-1.pdf">here</a>.

Technical Requirements

In [3]:
import json
import pandas as pd

with open('/home/jeremy/Documents/GitHub/mtb-data-research/msnbc-scraper/memory/WR-10-05-2025.json', 'r') as file:
    article_data = json.load(file)

df = pd.DataFrame(article_data)

Scraping news article data is one of the most basic forms of web data collection. Measures to avoid scraping can usually be circumvented by tailoring the bot's indexing criteria to the source website's class structure. Essentially, we need to actually tell the bot <i>where</i> to look for content on each outlet's article page.

#### Source Selection

Choosing the right sources to base product predictions on is the most importart part to remember about this method. MSNBC offers an slew of RSS Feeds in various categories straight from their website: https://www.cnbc.com/rss-feeds/. This is a reliable source for a general idea on what is popular in the Retail space. Travel and Media are two other RSS Feeds available from MSNBC that are related to MTB's product interests.

For more category-specific articles, outlets that report on the category must be researched. A few things to consider while searching are:

<ol><li>How often does the outlet upload articles?
    <li>How often are specific products written about?
    <li>Does the outlet follow a seasonal or regular schedule?
    <li>Where does this outlet source their information from?
    </ol>

#### Keyword Selection

The keyword detection program includes an algorithim that uses POS tagging (Part-of-Speech Tagging). This is the process of labeling each word in a sentence with its corresponding grammatical category.

There exists packages such as `stopwords`, `work_tokenize` and `FreeDist` from `nltk` that have built in mechanisms that can extract these keywords.

In addition to the keywords itself, LLM assistance would be extremely helpful in making decisions. Mountain Top Brand will do everything possible to use local, non-propriatary AI technilogies before considering third party AI servies via API calls.

### Data Example

The below example is keyword, frequency and keyword confidence data from a random article in the list. `keyword` is the phrase extracted, `frequency` represents how many times the words appears in the article, and `confidence` is a score between 0 and 1 indicating how relevant or meaningful that keyword is to the article's main subject.

In [9]:
# Extract the 'keywords' list from the first article
keywords_data = article_data[0]['keywords']

# Convert to a DataFrame
keywords_df = pd.DataFrame(keywords_data)

# Display the DataFrame
print(keywords_df)
print(pd.DataFrame(article_data['keywords'][0]))

           keyword  frequency  confidence
0            trump         28       0.188
1            miami         19       0.128
2          library         17       0.114
3             land         13       0.087
4          college          9       0.060
5            board          9       0.060
6             deal          6       0.040
7     presidential          6       0.040
8             dunn          6       0.040
9           office          6       0.040
10            dade          6       0.040
11            plan          6       0.040
12         station          6       0.040
13  administration          6       0.040
14           hotel          6       0.040


TypeError: list indices must be integers or slices, not str

<hr>

## Social Media Transcribing

Business Requirements

Where certain categories become popular, and trends start is on social media. Certain `#hashtags` can provide information on what product categories are popular at the current time. Using the social media algorithims from several smartphones, keywords can be extracted from feed data the same say article data was collected in the prevous section. 

This method includes some investment, but potential return is far greater. MTB would require sufficient R&D funding to purchase enough smartphones with a clear enough screen and loud enough speaker to record and transcribe audio successfully.

Technical Requirements

In 2025 there are many open source speech-to-text models we can use to transcribe the social media feed audio.

#### Data Example